In [1]:
print("hi")

hi


In [2]:
# ==========================
# Import Required Libraries
# ==========================
 
import os
from dotenv import load_dotenv
from typing import TypedDict
 
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

In [3]:

# ==========================
# Load Environment Variables
# ==========================
 
load_dotenv(override=True)
 
llm = ChatAnthropic(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT"),
    temperature=0
)
 
print("LLM Loaded Successfully!")

LLM Loaded Successfully!


In [4]:
#Step 4: Create State
#The graph stores all information inside the state.
# ==========================
# Graph State
# ==========================
 
class ExpenseState(TypedDict):
    question: str
    amount1: float
    amount2: float
    total: float
    tax_percent: float
    after_tax: float
    people: int
    per_person: float
    answer: str

Step 5: Create Tool Functions
These are the tools mentioned in your assignment.

In [5]:

# ==========================
# Tool 1
# Add Expenses
# ==========================
 
def add_expenses(a, b):
    return a + b
 
 
# ==========================
# Tool 2
# Apply Tax
# ==========================
 
def apply_tax(amount, tax_percent):
    return amount + (amount * tax_percent / 100)
 
 
# ==========================
# Tool 3
# Split Bill
# ==========================
 
def split_bill(amount, people):
    return amount / people


Step 6: Create Graph Nodes

In [6]:

#Node 1 → Addition
def add_node(state):
    total = add_expenses(state["amount1"], state["amount2"])
 
    return {
        "total": total
    }
#Node 2 → Tax
def tax_node(state):
 
    after_tax = apply_tax(
        state["total"],
        state["tax_percent"]
    )
 
    return {
        "after_tax": after_tax
    }
#Node 3 → Split
def split_node(state):
 
    split = split_bill(
        state["after_tax"],
        state["people"]
    )
 
    return {
        "per_person": split
    }
#Node 4 → Final Answer
def answer_node(state):
 
    answer = f"""
Total Expense : ₹{state['total']}
 
After {state['tax_percent']}% GST : ₹{state['after_tax']}
 
Split Among {state['people']} People
 
Each Person Pays : ₹{state['per_person']:.2f}
"""
 
    return {
        "answer": answer
    }

In [7]:
# ==========================
# Build LangGraph
# ==========================
 
builder = StateGraph(ExpenseState)
 
builder.add_node("add", add_node)
builder.add_node("tax", tax_node)
builder.add_node("split", split_node)
builder.add_node("answer", answer_node)
 
builder.add_edge(START, "add")
builder.add_edge("add", "tax")
builder.add_edge("tax", "split")
builder.add_edge("split", "answer")
builder.add_edge("answer", END)
 
graph = builder.compile()


In [8]:
#Step 8: Test 1
result = graph.invoke({
 
    "question":"What is the total?",
 
    "amount1":1200,
 
    "amount2":800,
 
    "tax_percent":18,
 
    "people":2
 
})
 
print(result["answer"])


Total Expense : ₹2000

After 18% GST : ₹2360.0

Split Among 2 People

Each Person Pays : ₹1180.00



In [9]:
#Step 9: Test 2
result = graph.invoke({
 
    "question":"Add expenses",
 
    "amount1":2000,
 
    "amount2":1500,
 
    "tax_percent":18,
 
    "people":3
 
})
 
print(result["answer"])


Total Expense : ₹3500

After 18% GST : ₹4130.0

Split Among 3 People

Each Person Pays : ₹1376.67




#Step 10 (Optional): Graph Structure 
# Expense Calculator Agent
 
START
   │
   ▼
Add Expenses
   │
   ▼
Apply Tax
   │
   ▼
Split Bill
   │
   ▼
Generate Final Answer
   │
   ▼
END
 